# Document Ingestion Pipeline

This notebook runs the full ingestion pipeline:
- **Load** documents (PDF, DOCX, XLSX, CSV, HTML, JSON, Images)
- **Chunk** text using RecursiveChunker or TableAwareChunker
- **Embed** with BGE-Large-EN-v1.5 (1024-dim)
- **Deduplicate** via hash-based detection
- **Store** in ChromaDB vector database

---

In [1]:
# Setup: add project root to path
import sys
import os
from pathlib import Path

# _setup.py lives in the same folder as this notebook.
# It sets PROJECT_ROOT, adds it to sys.path, and loads .env.
# If this import fails, add the notebooks/ folder to sys.path first:
#   sys.path.insert(0, '/path/to/financial-advisor/notebooks')
from _setup import PROJECT_ROOT

print(f'Project root: {PROJECT_ROOT}')
print(f'CWD: {os.getcwd()}')

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor
CWD: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor


## 1. Check Current Vector Store Status

In [2]:
from storage.chroma_store import ChromaStore, CHROMA_COLLECTION_FINANCIAL, CHROMA_COLLECTION_USER_DATA, CHROMA_COLLECTION_LEGACY

# Check all known collections
for name in [CHROMA_COLLECTION_FINANCIAL, CHROMA_COLLECTION_USER_DATA, CHROMA_COLLECTION_LEGACY]:
    try:
        store = ChromaStore(collection_name=name)
        print(f"  Collection '{name}': {store.count()} vectors")
    except Exception as e:
        print(f"  Collection '{name}': ERROR - {e}")

  Collection 'financial_data': 28154 vectors
  Collection 'user_data': 1617 vectors
  Collection 'financial_docs_cdc_hcp': 0 vectors


## 2. Ingest a Single File

Change `FILE_PATH` below to any supported document.

In [3]:
from ingestion import IngestionPipeline
from ingestion.config import IngestionConfig

# Configure the pipeline
config = IngestionConfig(
    chunk_size=512,
    chunk_overlap=64,
    dedup_enabled=True,
)

pipeline = IngestionPipeline(config=config)

print(f"Embedding model: {config.embed_model}")
print(f"Chunk size: {config.chunk_size}, Overlap: {config.chunk_overlap}")
print(f"Target collection: {config.chroma_collection}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model: BAAI/bge-m3
Chunk size: 512, Overlap: 64
Target collection: financial_data


In [4]:
# ── Ingest a single file ──────────────────────────────────────────────────
# Point this to any PDF, DOCX, XLSX, CSV, HTML, JSON, or image file:
FILE_PATH = str(PROJECT_ROOT / "data" / "collected_data" / "documents" / "ammc" / "ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf")

print(f"Ingesting: {Path(FILE_PATH).name}")
result = pipeline.ingest_file(FILE_PATH)

print(f"\n[OK] Result:")
print(f"  Documents: {result['documents']}")
print(f"  Chunks:    {result['chunks']}")
print(f"  Stored:    {result['stored']}")

Ingesting: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf

[OK] Result:
  Documents: 20
  Chunks:    92
  Stored:    92


## 3. Ingest an Entire Directory

Walks the directory recursively and ingests all supported files.

In [ ]:
# ── Ingest all files in a directory ───────────────────────────────────────
DIRECTORY = str(PROJECT_ROOT / "data" / "collected_data" / "documents" / "ammc")

print(f"Ingesting directory: {DIRECTORY}")
result = pipeline.ingest_directory(DIRECTORY)

print(f"\n[OK] Directory Result:")
print(f"  Documents: {result['documents']}")
print(f"  Chunks:    {result['chunks']}")
print(f"  Stored:    {result['stored']}")

## 4. Ingest into User Data Collection

User-uploaded docs go to a separate `user_data` collection.

In [5]:
from storage.chroma_store import CHROMA_COLLECTION_USER_DATA

user_pipeline = IngestionPipeline(collection_override=CHROMA_COLLECTION_USER_DATA)

# Replace with your own file path:
# result = user_pipeline.ingest_file("path/to/your/document.pdf")
# print(result)

print(f"User data collection: '{CHROMA_COLLECTION_USER_DATA}' has {user_pipeline.count()} vectors")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

User data collection: 'user_data' has 1617 vectors


## 5. Inspect Chunking (Debug)

See how a document gets chunked before embedding.

In [6]:
from ingestion.loaders.pdf_loader import PDFLoader
from ingestion.chunkers.recursive import RecursiveChunker

# Load a PDF
loader = PDFLoader()
records = loader.load(FILE_PATH)
print(f"Loaded {len(records)} text record(s) from the PDF")

if records:
    text, meta = records[0]
    print(f"\nFirst record preview ({len(text)} chars):")
    print(text[:500] + "...")
    print(f"\nMetadata: {meta}")

Loaded 20 text record(s) from the PDF

First record preview (109 chars):
Guide pédagogique
COPRENDRE LES OPCV

[TABLE]
Guide pédagogique
COPRENDRE LES OPCV |  | 
 |  | 

 |  |  |...

Metadata: {'source': 'ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf', 'doc_type': 'pdf', 'page': 1, 'file_path': 'C:\\Users\\lenovo\\Desktop\\Pfe 2026\\financial-advisor\\data\\collected_data\\documents\\ammc\\ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf'}


In [7]:
# Chunk the text
chunker = RecursiveChunker(chunk_size=512, chunk_overlap=64)

all_chunks = []
for text, meta in records:
    chunks = chunker.chunk(text, meta)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"\nChunk #1 ({len(all_chunks[0][0])} chars):")
print(all_chunks[0][0][:300])

Total chunks: 92

Chunk #1 (109 chars):
Guide pédagogique
COPRENDRE LES OPCV

[TABLE]
Guide pédagogique
COPRENDRE LES OPCV |  | 
 |  | 

 |  |  |


## 6. Verify Storage

Query ChromaDB to confirm the documents were stored correctly.

In [8]:
from ingestion.embeddings import Embedder

embedder = Embedder()
store = ChromaStore()

# Search for a term to verify ingestion
query = "OPCVM fonds investissement"
query_embedding = embedder.embed([query])[0]

results = store.collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Search results for: '{query}'\n")
for i, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
    print(f"--- Result {i+1} (distance: {dist:.4f}) ---")
    print(f"Source: {meta.get('source_file', 'unknown')}")
    print(f"Text: {doc[:200]}...\n")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Search results for: 'OPCVM fonds investissement'

--- Result 1 (distance: 0.2070) ---
Source: ammc_RMC__N3-Avril_2019pdf_d791b9be.pdf
Text: | OPCVM...

--- Result 2 (distance: 0.2923) ---
Source: ammc_Guide_de_linvestisseur_-_Comprendre_les_circuits_3de76862.pdf
Text: Guide de l’investisseur : Comprendre les circuits et adopter les bonnes pratiques
Zoom sur les OPCVM
C’est quoi ?
Un Organisme de Placement Collectif en Valeurs Mobilières, ou OPCVM, est un fonds
d’in...

--- Result 3 (distance: 0.2923) ---
Source: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf
Text: Guide pédagogique : Comprendre les OPCVM
II INVESTIR EN OPCVM
1 QUELS SONT LES AVANTAGES DE L’INVESTISSEMENT EN OPCVM ?
De nombreux avantages sont associés à l’investissement en OPCVM :
+ L a diversif...

